# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. `5 (2, 3)` — элемент во 2-й строке и 3-м столбце равен 5, размерность массива 2 на 3.
2. `1 8` — индекс первого максимального элемента (8) равен 1, само максимальное значение равно 8.
3. `99` — срез `a[::3]` возвращает представление (view), а не копию, поэтому изменение `b[0]` меняет `a[0]`.
4. `2 0.5` — условие `a > 3` дает два элемента (5 и 7), их сумма (количество) равна 2, а доля (mean) равна 2/4 = 0.5.
5. `(2, 3) 0.25` — форма массива нулей равна (2, 3), а `linspace(0, 1, 5)` создает 5 точек (0.0, 0.25, 0.5, 0.75, 1.0), где элемент с индексом 1 равен 0.25.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# База
# 1. Самый дорогой заказ
idx_max = amount.argmax()
print(f"1. Позиция: {idx_max}, Сумма: {amount[idx_max]} ₸, Клиент: {cust[idx_max]}, Город: {city[idx_max]}")
print(f"   Менеджеру: Самый крупный заказ №{idx_max} на сумму {amount[idx_max]:,.2f} ₸ был оформлен клиентом {cust[idx_max]} из города {city[idx_max]}.")

# 2. Выручка оплаченных и сумма неоплаченных
revenue_paid = amount[paid].sum()
revenue_unpaid = amount[~paid].sum()
print(f"\n2. Выручка оплаченных: {revenue_paid:,.2f} ₸, Сумма неоплаченных: {revenue_unpaid:,.2f} ₸")
print(f"   Менеджеру: Фактическая выручка составила {revenue_paid:,.2f} ₸, а потенциальная потеря от неоплаченных заказов — {revenue_unpaid:,.2f} ₸.")

# 3. Заказы в Шымкенте
shymkent_mask = (city == "Шымкент")
shymkent_count = shymkent_mask.sum()
shymkent_share = shymkent_mask.mean()
print(f"\n3. Шымкент: {shymkent_count} заказов, доля: {shymkent_share:.2%}")
print(f"   Менеджеру: На Шымкент приходится {shymkent_count} заказов, что составляет {shymkent_share:.1%} от общего объема покупок.")

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# Стандарт
# 4. Скидка 15% на оплаченные заказы > 50 000 ₸
discount_mask = paid & (amount > 50000)
lost_revenue_all = (amount[discount_mask] * 0.15).sum()

not_test_mask = (cust != "TEST")
discount_mask_no_test = paid & (amount > 50000) & not_test_mask
lost_revenue_no_test = (amount[discount_mask_no_test] * 0.15).sum()

print(4)
print(f"Со всеми заказами недополучим: {lost_revenue_all:,.2f} ₸")
print(f"Без тестов недополучим: {lost_revenue_no_test:,.2f} ₸")
print("   Менеджеру: Ответы различаются в несколько раз из-за тестовых заказов с аномально большими суммами от клиента TEST, которые искажают общую картину промоакции.")

# 5. Доля заказов, доставленных за 1–2 дня по городам
print("\n5. Доля быстрых доставок (1–2 дня) по городам:")
for c in np.unique(city):
    c_mask = (city == c)
    valid_days = c_mask & ~np.isnan(days)
    if valid_days.sum() > 0:
        fast_share = ((days >= 1) & (days <= 2) & valid_days).sum() / valid_days.sum()
        print(f"   - {c}: {fast_share:.2%}")
print("   Менеджеру: Логистика в регионах работает по-разному, наибольшая доля быстрых доставок наблюдается в крупных узлах.")

# 6. Исправление расчета среднего чека
correct_avg = amount[paid].mean()
print(f"\n6. Исправленный средний чек (по оплаченным заказам): {round(correct_avg):,} ₸")
print("   Менеджеру: Средний чек по оплаченным заказам составил корректную сумму, так как неоплаченные заказы не должны учитываться в выручке.")

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж
import time

# 7. Сравнение скорости циклом и через маску
start_time = time.time()
s_for = 0
for i in range(len(amount)):
    if paid[i]:
        s_for += amount[i]
time_for = time.time() - start_time

start_time = time.time()
s_mask = amount[paid].sum()
time_mask = time.time() - start_time

speedup = time_for / time_mask if time_mask > 0 else float('inf')
print(f"7. Цикл: {time_for:.6f} сек, Маска: {time_mask:.6f} сек. Маска быстрее в {speedup:.1f} раз(-а).")
print("   Менеджеру: Векторные операции NumPy работают в разы быстрее питоновских циклов за счет оптимизации под C.")

# 8. Моделирование A/B-теста
np.random.seed(42)
purchases_a = (np.random.rand(1000) < 0.05).sum()
purchases_b = (np.random.rand(1000) < 0.06).sum()
print(f"\n8. Разовый тест: Группа A (5%) = {purchases_a} покупок, Группа B (6%) = {purchases_b} покупок.")

b_better_count = 0
for _ in range(100):
    a_sim = (np.random.rand(1000) < 0.05).sum()
    b_sim = (np.random.rand(1000) < 0.06).sum()
    if b_sim > a_sim:
        b_better_count += 1

print(f"   В {b_better_count} тестах из 100 группа B показала результат лучше группы A.")
print("   Менеджеру: При текущем размере выборки (1000 пользователей) разницу в конверсии 1% уловить удается не всегда из-за высокой дисперсии. Для надежных выводов при таких малых дельтах выборку нужно увеличивать.")